In [3]:
%run generate_data.py

bigbasket_capstone.db, orders_raw.csv, products.csv created.


In [4]:
import sqlite3

conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

print("Database connected successfully!")

Database connected successfully!


In [5]:
cur.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table';
""")

tables = cur.fetchall()

print(tables)

[('products',), ('customers',), ('orders',), ('category_targets',)]


In [6]:
for table in ["products", "customers", "orders", "category_targets"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(table, ":", count)

products : 31
customers : 50
orders : 500
category_targets : 6


In [7]:
cur.execute("""
SELECT status, COUNT(*) AS order_count
FROM orders
GROUP BY status;
""")

status_counts = cur.fetchall()

print(status_counts)

[('Cancelled', 42), ('Delivered', 434), ('Pending', 24)]


In [8]:
verification_text = """-- BigBasket Capstone - Part 1 Verification

-- Row counts
-- products: 31
-- customers: 50
-- orders: 500
-- category_targets: 6

-- Status counts
-- Delivered: 434
-- Cancelled: 42
-- Pending: 24
"""

with open("verify.sql", "w") as f:
    f.write(verification_text)

print("verify.sql created successfully.")

verify.sql created successfully.


In [9]:
cur.execute("""
SELECT o.order_id,
       o.order_date,
       o.amount_inr,
       c.name,
       c.city
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE c.city = 'Bengaluru';
""")

results = cur.fetchall()

for row in results[:10]:
    print(row)

(2, '2026-05-22', 60, 'Ayaan', 'Bengaluru')
(3, '2026-06-30', 150, 'Yash', 'Bengaluru')
(11, '2026-04-03', 300, 'Tara', 'Bengaluru')
(16, '2026-02-10', 120, 'Ayaan', 'Bengaluru')
(20, '2026-03-03', 45, 'Siya', 'Bengaluru')
(22, '2026-05-16', 270, 'Pooja', 'Bengaluru')
(25, '2026-02-26', 225, 'Pooja', 'Bengaluru')
(30, '2026-01-16', 85, 'Manish', 'Bengaluru')
(31, '2026-05-18', 168, 'Myra', 'Bengaluru')
(34, '2026-05-05', 360, 'Myra', 'Bengaluru')


In [10]:
cur.execute("""
SELECT DISTINCT category
FROM products;
""")

categories = cur.fetchall()

for row in categories:
    print(row[0])

Fruits & Vegetables
Dairy & Eggs
Snacks & Beverages
Personal Care
Household Essentials
Bakery


In [11]:
cur.execute("""
SELECT order_id,
       order_date,
       amount_inr
FROM orders
ORDER BY amount_inr DESC
LIMIT 5;
""")

top_orders = cur.fetchall()

for row in top_orders:
    print(row)

(214, '2026-01-15', 1100)
(396, '2026-02-08', 900)
(232, '2026-02-08', 750)
(288, '2026-04-05', 725)
(152, '2026-01-26', 720)


In [12]:
cur.execute("""
SELECT COUNT(*) AS total_orders
FROM orders;
""")

result = cur.fetchone()

print(result)

(500,)


In [13]:
cur.execute("""
SELECT order_id,
       payment_mode,
       amount_inr
FROM orders
WHERE payment_mode IN ('UPI', 'Wallet');
""")

results = cur.fetchall()

for row in results[:10]:
    print(row)

(1, 'UPI', 175)
(3, 'UPI', 150)
(4, 'UPI', 30)
(7, 'Wallet', 396)
(10, 'Wallet', 60)
(12, 'UPI', 80)
(15, 'Wallet', 65)
(19, 'UPI', 225)
(20, 'Wallet', 45)
(22, 'Wallet', 270)


In [14]:
cur.execute("""
SELECT order_id,
       amount_inr
FROM orders
WHERE amount_inr BETWEEN 100 AND 500;
""")

results = cur.fetchall()

for row in results[:10]:
    print(row)

(1, 175)
(3, 150)
(5, 220)
(7, 396)
(9, 267)
(11, 300)
(13, 140)
(14, 300)
(16, 120)
(17, 450)


In [15]:
cur.execute("""
SELECT order_id,
       amount_inr
FROM orders
WHERE amount_inr NOT BETWEEN 100 AND 500;
""")

results = cur.fetchall()

for row in results[:10]:
    print(row)

(2, 60)
(4, 30)
(6, 65)
(8, 80)
(10, 60)
(12, 80)
(15, 65)
(20, 45)
(21, 20)
(24, 50)


In [16]:
cur.execute("""
SELECT order_id,
       rating
FROM orders
WHERE rating IS NULL;
""")

results = cur.fetchall()

print("Number of orders with missing rating:", len(results))

for row in results[:10]:
    print(row)

Number of orders with missing rating: 66
(11, None)
(12, None)
(14, None)
(20, None)
(33, None)
(55, None)
(61, None)
(73, None)
(77, None)
(84, None)


In [17]:
# 02_aggregation_joins.sql
cur.execute("""
SELECT p.category,
       COUNT(*) AS order_count,
       SUM(o.amount_inr) AS total_revenue,
       AVG(o.amount_inr) AS avg_revenue
FROM orders o
INNER JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category;
""")

results = cur.fetchall()

for row in results:
    print(row)

('Bakery', 67, 15410, 230.0)
('Dairy & Eggs', 66, 14090, 213.4848484848485)
('Fruits & Vegetables', 73, 9790, 134.1095890410959)
('Household Essentials', 79, 21715, 274.873417721519)
('Personal Care', 66, 16382, 248.21212121212122)
('Snacks & Beverages', 83, 10895, 131.26506024096386)


In [18]:
cur.execute("""
SELECT p.category,
       COUNT(*) AS order_count,
       SUM(o.amount_inr) AS total_revenue,
       AVG(o.amount_inr) AS avg_revenue
FROM orders o
INNER JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category
HAVING total_revenue > 10000;
""")

results = cur.fetchall()

for row in results:
    print(row)

('Bakery', 67, 15410, 230.0)
('Dairy & Eggs', 66, 14090, 213.4848484848485)
('Household Essentials', 79, 21715, 274.873417721519)
('Personal Care', 66, 16382, 248.21212121212122)
('Snacks & Beverages', 83, 10895, 131.26506024096386)


In [19]:
cur.execute("""
SELECT p.product_id,
       p.product_name,
       COUNT(o.order_id) AS total_orders
FROM products p
LEFT JOIN orders o
    ON p.product_id = o.product_id
GROUP BY p.product_id, p.product_name
ORDER BY total_orders ASC, p.product_id;
""")

results = cur.fetchall()

for row in results:
    print(row)

(31, 'Premium Face Cream 50g', 0)
(16, 'Shampoo 340ml', 3)
(4, 'Apple 1kg', 6)
(14, 'Fruit Juice 1L', 7)
(20, 'Face Wash 100g', 7)
(8, 'Eggs (12pc)', 8)
(10, 'Butter 100g', 12)
(19, 'Hand Wash 250ml', 12)
(1, 'Banana 1kg', 14)
(15, 'Namkeen 200g', 14)
(23, 'Floor Cleaner 1L', 14)
(27, 'Croissant (2pc)', 14)
(7, 'Paneer 200g', 15)
(13, 'Biscuit Pack', 15)
(24, 'Toilet Cleaner 500ml', 15)
(28, 'Muffin Pack (4pc)', 15)
(30, 'Cookies 200g', 15)
(5, 'Spinach Bunch', 16)
(17, 'Toothpaste 150g', 16)
(9, 'Curd 400g', 17)
(29, 'Cake Slice', 18)
(3, 'Onion 1kg', 19)
(26, 'Bread Loaf', 19)
(21, 'Dish Wash Bar', 21)
(25, 'Garbage Bags (30pc)', 22)
(22, 'Detergent 1kg', 24)
(2, 'Tomato 1kg', 25)
(6, 'Toned Milk 1L', 26)
(11, 'Potato Chips 90g', 27)
(12, 'Cola 750ml', 32)
(18, 'Soap Bar 125g', 32)


In [20]:
cur.execute("""
SELECT product_id,
       product_name,
       total_revenue,
       CASE
           WHEN total_revenue >= 3000 THEN 'High'
           WHEN total_revenue >= 1000 THEN 'Medium'
           ELSE 'Low'
       END AS revenue_tier
FROM (
    SELECT p.product_id,
           p.product_name,
           COALESCE(
               SUM(
                   CASE
                       WHEN o.status = 'Delivered'
                       THEN o.amount_inr
                       ELSE 0
                   END
               ), 0
           ) AS total_revenue
    FROM products p
    LEFT JOIN orders o
        ON p.product_id = o.product_id
    GROUP BY p.product_id, p.product_name
);
""")

results = cur.fetchall()

for row in results:
    print(row)

(1, 'Banana 1kg', 2050, 'Medium')
(2, 'Tomato 1kg', 2640, 'Medium')
(3, 'Onion 1kg', 1960, 'Medium')
(4, 'Apple 1kg', 2340, 'Medium')
(5, 'Spinach Bunch', 800, 'Low')
(6, 'Toned Milk 1L', 3780, 'High')
(7, 'Paneer 200g', 4770, 'High')
(8, 'Eggs (12pc)', 1260, 'Medium')
(9, 'Curd 400g', 2520, 'Medium')
(10, 'Butter 100g', 1760, 'Medium')
(11, 'Potato Chips 90g', 1890, 'Medium')
(12, 'Cola 750ml', 3780, 'High')
(13, 'Biscuit Pack', 1295, 'Medium')
(14, 'Fruit Juice 1L', 1650, 'Medium')
(15, 'Namkeen 200g', 2280, 'Medium')
(16, 'Shampoo 340ml', 1980, 'Medium')
(17, 'Toothpaste 150g', 4655, 'High')
(18, 'Soap Bar 125g', 3480, 'High')
(19, 'Hand Wash 250ml', 3267, 'High')
(20, 'Face Wash 100g', 3000, 'High')
(21, 'Dish Wash Bar', 1080, 'Medium')
(22, 'Detergent 1kg', 7670, 'High')
(23, 'Floor Cleaner 1L', 5655, 'High')
(24, 'Toilet Cleaner 500ml', 3560, 'High')
(25, 'Garbage Bags (30pc)', 3750, 'High')
(26, 'Bread Loaf', 1395, 'Medium')
(27, 'Croissant (2pc)', 3640, 'High')
(28, 'Muffin Pac

In [21]:
cur.execute("""
SELECT p.category AS category,
       strftime('%Y-%m', o.order_date) AS month,
       COUNT(*) AS order_count,
       SUM(o.amount_inr) AS total_revenue,
       AVG(o.amount_inr) AS avg_revenue
FROM orders o
JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category,
         strftime('%Y-%m', o.order_date)
ORDER BY p.category, month;
""")

results = cur.fetchall()

print("Number of rows:", len(results))

for row in results[:10]:
    print(row)

Number of rows: 36
('Bakery', '2026-01', 6, 735, 122.5)
('Bakery', '2026-02', 8, 2440, 305.0)
('Bakery', '2026-03', 11, 3090, 280.90909090909093)
('Bakery', '2026-04', 13, 3115, 239.6153846153846)
('Bakery', '2026-05', 15, 3695, 246.33333333333334)
('Bakery', '2026-06', 14, 2335, 166.78571428571428)
('Dairy & Eggs', '2026-01', 9, 2035, 226.11111111111111)
('Dairy & Eggs', '2026-02', 8, 1935, 241.875)
('Dairy & Eggs', '2026-03', 15, 3163, 210.86666666666667)
('Dairy & Eggs', '2026-04', 9, 1650, 183.33333333333334)


In [22]:
cur.execute("""
WITH category_revenue AS (
    SELECT p.category,
           SUM(o.amount_inr) AS total_revenue
    FROM orders o
    JOIN products p
        ON o.product_id = p.product_id
    WHERE o.status = 'Delivered'
    GROUP BY p.category
)
SELECT cr.category,
       cr.total_revenue,
       t.target_revenue_inr,
       t.target_revenue_inr - cr.total_revenue AS variance,
       ((cr.total_revenue - t.target_revenue_inr) * 100.0)
           / t.target_revenue_inr AS percentage_variance,
       CASE
           WHEN cr.total_revenue >= t.target_revenue_inr
               THEN 'Above Target'
           WHEN ((cr.total_revenue - t.target_revenue_inr) * 100.0)
                    / t.target_revenue_inr >= -15
               THEN 'Below Target - Watch'
           ELSE 'Below Target - Critical'
       END AS target_status
FROM category_revenue cr
JOIN category_targets t
    ON cr.category = t.category;
""")

results = cur.fetchall()

for row in results:
    print(row)

('Bakery', 15410, 12000, -3410, 28.416666666666668, 'Above Target')
('Dairy & Eggs', 14090, 16500, 2410, -14.606060606060606, 'Below Target - Watch')
('Fruits & Vegetables', 9790, 12000, 2210, -18.416666666666668, 'Below Target - Critical')
('Household Essentials', 21715, 17000, -4715, 27.735294117647058, 'Above Target')
('Personal Care', 16382, 15500, -882, 5.690322580645161, 'Above Target')
('Snacks & Beverages', 10895, 13000, 2105, -16.192307692307693, 'Below Target - Critical')


In [23]:
cur.execute("""
WITH category_revenue AS (
    SELECT p.category,
           SUM(o.amount_inr) AS total_revenue
    FROM orders o
    JOIN products p
        ON o.product_id = p.product_id
    WHERE o.status = 'Delivered'
    GROUP BY p.category
)
SELECT cr.category,
       cr.total_revenue,
       t.target_revenue_inr,
       t.target_revenue_inr - cr.total_revenue AS variance,
       ((cr.total_revenue - t.target_revenue_inr) * 100.0)
           / t.target_revenue_inr AS percentage_variance,
       CASE
           WHEN cr.total_revenue >= t.target_revenue_inr
               THEN 'Above Target'
           WHEN ((cr.total_revenue - t.target_revenue_inr) * 100.0)
                    / t.target_revenue_inr >= -15
               THEN 'Below Target - Watch'
           ELSE 'Below Target - Critical'
       END AS target_status
FROM category_revenue cr
JOIN category_targets t
    ON cr.category = t.category;
""")

results = cur.fetchall()

for row in results:
    print(row)

('Bakery', 15410, 12000, -3410, 28.416666666666668, 'Above Target')
('Dairy & Eggs', 14090, 16500, 2410, -14.606060606060606, 'Below Target - Watch')
('Fruits & Vegetables', 9790, 12000, 2210, -18.416666666666668, 'Below Target - Critical')
('Household Essentials', 21715, 17000, -4715, 27.735294117647058, 'Above Target')
('Personal Care', 16382, 15500, -882, 5.690322580645161, 'Above Target')
('Snacks & Beverages', 10895, 13000, 2105, -16.192307692307693, 'Below Target - Critical')


In [24]:
import csv

monthly_query = """
SELECT p.category AS category,
       strftime('%Y-%m', o.order_date) AS month,
       COUNT(*) AS order_count,
       SUM(o.amount_inr) AS total_revenue,
       AVG(o.amount_inr) AS avg_revenue
FROM orders o
JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category, strftime('%Y-%m', o.order_date)
ORDER BY p.category, month;
"""

cur.execute(monthly_query)

rows = cur.fetchall()
columns = [description[0] for description in cur.description]

with open("monthly_category_revenue.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    writer.writerows(rows)

print("monthly_category_revenue.csv created successfully.")

monthly_category_revenue.csv created successfully.


In [25]:
print(len(rows))
print(columns)
print(sum(row[3] for row in rows))

36
['category', 'month', 'order_count', 'total_revenue', 'avg_revenue']
88282


In [26]:
ai_log_text = """# AI Assistance Log

## Prompt 1 — Monthly Category Revenue Query

### Role
You are an experienced SQLite data analyst and SQL tutor.

### Context
I am working on a deterministic BigBasket-style SQLite database with orders and products. The orders table contains order_date, amount_inr, status, and product_id. The products table contains product_id and category.

### Task
Help me write a monthly-by-category Delivered revenue query with the following columns:
category, month, order_count, total_revenue, avg_revenue.

### Constraints
Use SQLite strftime('%Y-%m', order_date) for the month.
Join orders and products using product_id.
Filter only status = 'Delivered'.
Group by category and month.
Order by category and month.
Do not modify the underlying data.

### Format
Return one runnable SQLite query and a short explanation of each clause.

## Verification Performed

I ran the generated query against the SQLite database and verified that it returned:

- 36 rows
- Columns: category, month, order_count, total_revenue, avg_revenue
- Grand total Delivered revenue: ₹88,282

The monthly results were then exported to monthly_category_revenue.csv and the exported data was verified using Python.
"""

with open("ai_log.md", "w", encoding="utf-8") as f:
    f.write(ai_log_text)

print("ai_log.md created successfully.")

ai_log.md created successfully.


In [27]:
import os

for file in [
    "generate_data.py",
    "bigbasket_capstone.db",
    "orders_raw.csv",
    "products.csv",
    "verify.sql",
    "01_foundations.sql",
    "02_aggregation_joins.sql",
    "03_reporting.sql",
    "monthly_category_revenue.csv",
    "ai_log.md"
]:
    print(file, "->", os.path.exists(file))

generate_data.py -> True
bigbasket_capstone.db -> True
orders_raw.csv -> True
products.csv -> True
verify.sql -> True
01_foundations.sql -> True
02_aggregation_joins.sql -> True
03_reporting.sql -> True
monthly_category_revenue.csv -> True
ai_log.md -> True


In [4]:
import pandas as pd
pd.read_sql_query("""
SELECT 
    p.category,
    SUM(o.amount_inr) AS total_revenue
FROM orders o
JOIN products p
    ON o.product_id = p.product_id
WHERE o.status = 'Delivered'
GROUP BY p.category
ORDER BY p.category;
""", conn)

NameError: name 'conn' is not defined

In [10]:
%%writefile DATA_STORY.md
# BigBasket Revenue Performance Data Story

Overwriting DATA_STORY.md


In [11]:
%%writefile DATA_STORY.md
# BigBasket Revenue Performance Data Story

## Overview

The dashboard analyzes delivered revenue performance across six BigBasket-style product categories from January to June 2026. Total delivered revenue was ₹88,282 from 434 delivered orders, with an average order value of ₹203.41.

## Category Performance

### Above Target

- **Household Essentials:** Revenue was ₹21,715 against a target of ₹17,000, which is ₹4,715 above target (27.74% above target).
- **Personal Care:** Revenue was ₹16,382 against a target of ₹15,500, which is ₹882 above target (5.69% above target).
- **Bakery:** Revenue was ₹15,410 against a target of ₹12,000, which is ₹3,410 above target (28.42% above target).

### Below Target - Watch

- **Dairy & Eggs:** Revenue was ₹14,090 against a target of ₹16,500, which is ₹2,410 below target (14.61% below target).

### Below Target - Critical

- **Snacks & Beverages:** Revenue was ₹10,895 against a target of ₹13,000, which is ₹2,105 below target (16.19% below target).
- **Fruits & Vegetables:** Revenue was ₹9,790 against a target of ₹12,000, which is ₹2,210 below target (18.42% below target).

## Recommendations

1. **Prioritize Fruits & Vegetables and Snacks & Beverages for improvement**, because both are classified as Below Target - Critical and have the largest percentage gaps from their targets.

2. **Maintain the strong performance of Household Essentials and Bakery while investigating what is driving their higher revenue**, since they are the two categories with the largest positive revenue gaps above target.

Overwriting DATA_STORY.md


In [12]:
with open("DATA_STORY.md", "r", encoding="utf-8") as f:
    print(f.read())

# BigBasket Revenue Performance Data Story

## Overview

The dashboard analyzes delivered revenue performance across six BigBasket-style product categories from January to June 2026. Total delivered revenue was ₹88,282 from 434 delivered orders, with an average order value of ₹203.41.

## Category Performance

### Above Target

- **Household Essentials:** Revenue was ₹21,715 against a target of ₹17,000, which is ₹4,715 above target (27.74% above target).
- **Personal Care:** Revenue was ₹16,382 against a target of ₹15,500, which is ₹882 above target (5.69% above target).
- **Bakery:** Revenue was ₹15,410 against a target of ₹12,000, which is ₹3,410 above target (28.42% above target).

### Below Target - Watch

- **Dairy & Eggs:** Revenue was ₹14,090 against a target of ₹16,500, which is ₹2,410 below target (14.61% below target).

### Below Target - Critical

- **Snacks & Beverages:** Revenue was ₹10,895 against a target of ₹13,000, which is ₹2,105 below target (16.19% below target).
- 